In [ ]:
# 📊 Exploratory Data Analysis - Retail Customer Churn

**Objective:** Analyze customer data to understand patterns leading to churn and prepare for ML modeling.

**Dataset:** 4,374 retail customers with 52 features (RFM metrics, demographics, behavior patterns)

: 

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', None)

: 

In [ ]:
# Load data
df = pd.read_csv('../data/raw/retail_customers_COMPLETE_CATEGORICAL.csv')
print(f"Dataset shape: {df.shape}")
print(f"Total customers: {df.shape[0]:,}")
print(f"Total features: {df.shape[1]}")

## 1. Dataset Overview

In [ ]:
# Data types overview
print("="*60)
print("DATA TYPES SUMMARY")
print("="*60)
print(f"\nNumerical features: {df.select_dtypes(include=['int64', 'float64']).shape[1]}")
print(f"Categorical features: {df.select_dtypes(include=['object']).shape[1]}")
print("\n" + "="*60)
df.info()

In [ ]:
# Statistical summary of numerical features
df.describe().T.round(2)

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

if len(missing_df) > 0:
    print("Columns with missing values:")
    print(missing_df)
    
    # Visualize missing values
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['Percentage'].plot(kind='bar', ax=ax, color='coral')
    ax.set_ylabel('Percentage Missing')
    ax.set_title('Missing Values by Column')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found!")

## 2. Target Variable Analysis (Churn)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(churn_counts.index.map({0: 'No Churn', 1: 'Churn'}), 
            churn_counts.values, color=colors)
axes[0].set_title('Churn Distribution (Count)')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Percentage pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'], 
            autopct='%1.1f%%', colors=colors, explode=[0, 0.05])
axes[1].set_title('Churn Distribution (%)')

plt.tight_layout()
plt.show()

# Print summary
print(f"\nChurn Rate: {df['Churn'].mean()*100:.2f}%")
print(f"Class Imbalance Ratio: 1:{churn_counts[0]/churn_counts[1]:.1f}")

## 3. Numerical Features Analysis

In [ ]:
# RFM Metrics Distribution by Churn Status
rfm_cols = ['Recency', 'Frequency', 'MonetaryTotal', 'MonetaryAvg']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(rfm_cols):
    for churn_val, color, label in [(0, '#2ecc71', 'No Churn'), (1, '#e74c3c', 'Churn')]:
        data = df[df['Churn'] == churn_val][col]
        axes[idx].hist(data, bins=30, alpha=0.6, label=label, color=color, density=True)
    axes[idx].set_title(f'{col} Distribution by Churn')
    axes[idx].set_xlabel(col)
    axes[idx].legend()

plt.tight_layout()
plt.suptitle('RFM Metrics Analysis', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Boxplots for key metrics by Churn
key_metrics = ['Recency', 'Frequency', 'MonetaryTotal', 'CustomerTenureDays', 
               'UniqueProducts', 'SatisfactionScore']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(key_metrics):
    if col in df.columns:
        df.boxplot(column=col, by='Churn', ax=axes[idx])
        axes[idx].set_title(f'{col}')
        axes[idx].set_xlabel('Churn')

plt.suptitle('Key Metrics by Churn Status', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Categorical Features Analysis

In [ ]:
# Key categorical features
cat_features = ['RFMSegment', 'ChurnRiskCategory', 'CustomerType', 'SpendingCategory']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(cat_features):
    if col in df.columns:
        # Calculate churn rate per category
        churn_by_cat = df.groupby(col)['Churn'].agg(['mean', 'count']).reset_index()
        churn_by_cat.columns = [col, 'ChurnRate', 'Count']
        churn_by_cat = churn_by_cat.sort_values('ChurnRate', ascending=True)
        
        # Plot
        bars = axes[idx].barh(churn_by_cat[col], churn_by_cat['ChurnRate']*100, 
                              color=plt.cm.RdYlGn_r(churn_by_cat['ChurnRate']))
        axes[idx].set_xlabel('Churn Rate (%)')
        axes[idx].set_title(f'Churn Rate by {col}')
        axes[idx].axvline(x=df['Churn'].mean()*100, color='red', linestyle='--', 
                         label=f'Overall: {df["Churn"].mean()*100:.1f}%')
        axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Demographics analysis
demo_features = ['AgeCategory', 'Gender', 'Region', 'LoyaltyLevel']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(demo_features):
    if col in df.columns:
        ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
        ct.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'])
        axes[idx].set_title(f'Churn by {col}')
        axes[idx].set_ylabel('Percentage')
        axes[idx].legend(['No Churn', 'Churn'])
        axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Select most important features for correlation
important_cols = ['Recency', 'Frequency', 'MonetaryTotal', 'MonetaryAvg', 
                  'CustomerTenureDays', 'UniqueProducts', 'TotalTransactions',
                  'SatisfactionScore', 'SupportTicketsCount', 'ReturnRatio', 'Churn']

# Filter columns that exist
important_cols = [c for c in important_cols if c in df.columns]

corr_matrix = df[important_cols].corr()

# Heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix - Key Features vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top features correlated with Churn
churn_corr = df[numeric_cols].corr()['Churn'].drop('Churn').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in churn_corr.head(15)]
churn_corr.head(15).plot(kind='barh', color=colors)
plt.xlabel('Correlation with Churn')
plt.title('Top 15 Features Correlated with Churn', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nTop 10 features most correlated with Churn:")
print(churn_corr.head(10).round(3).to_string())

## 6. Outlier Detection

In [ ]:
# Check SatisfactionScore for outliers
if 'SatisfactionScore' in df.columns:
    print("SatisfactionScore Analysis:")
    print(f"Min: {df['SatisfactionScore'].min()}")
    print(f"Max: {df['SatisfactionScore'].max()}")
    print(f"Mean: {df['SatisfactionScore'].mean():.2f}")
    print(f"\nValue counts for potential outliers:")
    score_counts = df['SatisfactionScore'].value_counts().sort_index()
    print(f"Values <= 0: {(df['SatisfactionScore'] <= 0).sum()}")
    print(f"Values >= 90: {(df['SatisfactionScore'] >= 90).sum()}")
    
    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(df['SatisfactionScore'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_title('SatisfactionScore Distribution')
    axes[0].set_xlabel('Score')
    axes[0].axvline(x=0, color='red', linestyle='--', label='Potential outliers')
    
    # Boxplot
    df.boxplot(column='SatisfactionScore', ax=axes[1])
    axes[1].set_title('SatisfactionScore Boxplot')
    
    plt.tight_layout()
    plt.show()

## 7. Key Insights Summary

In [ ]:
# Generate summary statistics
print("="*60)
print("📊 EDA KEY INSIGHTS SUMMARY")
print("="*60)

print(f"\n1. DATASET SIZE: {df.shape[0]:,} customers, {df.shape[1]} features")
print(f"\n2. CHURN RATE: {df['Churn'].mean()*100:.2f}%")

print(f"\n3. CLASS IMBALANCE:")
churn_dist = df['Churn'].value_counts()
print(f"   - No Churn: {churn_dist[0]:,} ({churn_dist[0]/len(df)*100:.1f}%)")
print(f"   - Churn: {churn_dist[1]:,} ({churn_dist[1]/len(df)*100:.1f}%)")

print(f"\n4. TOP CHURN RISK SEGMENTS:")
if 'ChurnRiskCategory' in df.columns:
    risk_churn = df.groupby('ChurnRiskCategory')['Churn'].mean().sort_values(ascending=False)
    for cat, rate in risk_churn.head(3).items():
        print(f"   - {cat}: {rate*100:.1f}% churn rate")

print(f"\n5. TOP CORRELATED FEATURES:")
for feat, corr in churn_corr.head(5).items():
    direction = "↑" if corr > 0 else "↓"
    print(f"   - {feat}: {corr:.3f} {direction}")

print("\n" + "="*60)
print("📋 RECOMMENDATIONS FOR MODELING:")
print("="*60)
print("• Handle class imbalance (use class_weight='balanced' or SMOTE)")
print("• Clean SatisfactionScore outliers before training")
print("• Focus on RFM features (high correlation with Churn)")
print("• Consider ChurnRiskCategory as a strong predictor")
print("• Use tree-based models (handle mixed features well)")